# Day 5 - Neural Networks and ANN with Keras (RetailMind AI)

**Goal:** build a Keras ANN for churn and compare it with the best classical model from Phase 1 (`churn_v4`). Did the extra depth actually help?

## 1. Setup

In [1]:
import sys
sys.path.append("..")

import joblib
import numpy as np
import pandas as pd
from tensorflow import keras
from sklearn.model_selection import train_test_split

from src.config import MODEL_PATH
from src.data_loader import load_data, split_features_target
from src.preprocessing import build_preprocessor
from src.evaluate import get_metrics, get_metrics_from_proba

keras.utils.set_random_seed(42)   # same results every run

## 2. Warm-up on MNIST
A quick look at how Keras works: layers, activations and a training loop. MNIST is 28x28 pictures of handwritten digits (0-9).

In [2]:
(X_tr, y_tr), (X_te, y_te) = keras.datasets.mnist.load_data()
X_tr, X_te = X_tr / 255.0, X_te / 255.0     # scale pixels from 0-255 to 0-1

demo = keras.Sequential([
    keras.Input(shape=(28, 28)),
    keras.layers.Flatten(),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(10, activation="softmax"),
])
demo.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
demo.fit(X_tr, y_tr, epochs=3, batch_size=128, validation_split=0.1, verbose=2)

loss, acc = demo.evaluate(X_te, y_te, verbose=0)
print("MNIST test accuracy:", round(acc, 4))

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Epoch 1/3
422/422 - 5s - 11ms/step - accuracy: 0.8796 - loss: 0.4430 - val_accuracy: 0.9492 - val_loss: 0.1980
Epoch 2/3
422/422 - 2s - 5ms/step - accuracy: 0.9399 - loss: 0.2100 - val_accuracy: 0.9602 - val_loss: 0.1505
Epoch 3/3
422/422 - 2s - 5ms/step - accuracy: 0.9533 - loss: 0.1604 - val_accuracy: 0.9623 - val_loss: 0.1283
MNIST test accuracy: 0.957


## 3. Prepare the churn data
Same data and the same split as Days 1-4, so the comparison is fair. Neural networks need **normalised inputs**, and Day 1's preprocessing already scales the numbers and one-hot encodes the text.

In [3]:
df = load_data()
X, y = split_features_target(df)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

preprocessor = build_preprocessor(X_train)
X_train_p = preprocessor.fit_transform(X_train)
X_test_p = preprocessor.transform(X_test)

# Keras wants plain arrays
if hasattr(X_train_p, "toarray"):
    X_train_p, X_test_p = X_train_p.toarray(), X_test_p.toarray()
X_train_p = X_train_p.astype("float32")
X_test_p = X_test_p.astype("float32")
print("Input shape:", X_train_p.shape)

Input shape: (5634, 45)


## 4. Build a small ANN
normalised inputs, then dense layers, then a sigmoid output (one number between 0 and 1 = chance of churn).

In [4]:
ann = keras.Sequential([
    keras.Input(shape=(X_train_p.shape[1],)),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dense(16, activation="relu"),
    keras.layers.Dense(1, activation="sigmoid"),
])
ann.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
ann.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓
┃ Layer (type)             ┃ Output Shape      ┃   Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩
│ dense_2 (Dense)          │ (None, 32)        │     1,472 │
├──────────────────────────┼───────────────────┼───────────┤
│ dense_3 (Dense)          │ (None, 16)        │       528 │
├──────────────────────────┼───────────────────┼───────────┤
│ dense_4 (Dense)          │ (None, 1)         │        17 │
└──────────────────────────┴───────────────────┴───────────┘

 Total params: 2,017 (7.88 KB)

 Trainable params: 2,017 (7.88 KB)

 Non-trainable params: 0 (0.00 B)

## 5. Train with early stopping
- **Early stopping** watches the validation loss and stops when it stops improving, so the network doesn't overfit this small tabular dataset.
- **class_weight** counts a missed churner as a bigger mistake (like `class_weight="balanced"` in Phase 1), because recall matters most.

In [5]:
weight_for_churn = (y_train == 0).sum() / (y_train == 1).sum()
class_weight = {0: 1.0, 1: float(weight_for_churn)}

early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)

history = ann.fit(
    X_train_p, y_train.to_numpy(),
    validation_split=0.2, epochs=100, batch_size=32,
    class_weight=class_weight, callbacks=[early_stop], verbose=0,
)
print("Stopped after", len(history.history["loss"]), "epochs")

Stopped after 12 epochs


Training loss vs validation loss for the last few epochs. If validation loss starts rising while training loss keeps falling, that's overfitting.

In [6]:
pd.DataFrame(history.history)[["loss", "val_loss"]].tail()

,loss,val_loss
7,0.683908,0.572898
8,0.681124,0.574969
9,0.678747,0.576753
10,0.676376,0.576991
11,0.673780,0.577835


## 6. Benchmark against the Phase 1 champion
Both are scored on the same hold-out customers. (The ANN gets the plain Day 1 features; the champion is the saved `churn_v4` model.)

In [7]:
proba = ann.predict(X_test_p, verbose=0).ravel()
ann_scores = get_metrics_from_proba(y_test, proba)

champion = joblib.load(MODEL_PATH.parent / "churn_v4.joblib")
champ_scores = get_metrics(champion, X_test, y_test)

comparison = pd.DataFrame({
    "Phase 1 champion (churn_v4)": champ_scores,
    "Keras ANN": ann_scores,
}).T
comparison

,Accuracy,Precision,Recall,F1,ROC-AUC
Phase 1 champion (churn_v4),0.7594,0.5318,0.7834,0.6335,0.8447
Keras ANN,0.7211,0.4851,0.8262,0.6113,0.8366


In [8]:
diff = round(ann_scores["ROC-AUC"] - champ_scores["ROC-AUC"], 4)
print("ROC-AUC difference (ANN - champion):", diff)

if diff > 0.005:
    print("The ANN is better, so depth helped here.")
elif diff < -0.005:
    print("The classical champion is better, so depth did not help here.")
else:
    print("About the same, so depth did not clearly help here.")

ROC-AUC difference (ANN - champion): -0.0081
The classical champion is better, so depth did not help here.


**Note:** neural networks usually shine on images, text and very large datasets. On a small table of about 7,000 customers, a tuned Random Forest or XGBoost is often just as good, and it is faster to train and easier to explain.

## 7. Save the model
The ANN is saved together with its preprocessor, because new customers must be scaled and encoded the same way.

In [9]:
ann.save(MODEL_PATH.parent / "churn_ann.keras")
joblib.dump(preprocessor, MODEL_PATH.parent / "churn_ann_preprocessor.joblib")
print("Saved churn_ann.keras and churn_ann_preprocessor.joblib")

Saved churn_ann.keras and churn_ann_preprocessor.joblib
